# Phase 6 (companion) — wider baseline panel (kNN / GMM / position-density)

> Issue #27 · write-up: `backend/docs/ml/07-train.md` §4c. Firewall-clean: fit on TRAIN-normal,
> score the SAME val eval set, **TEST sealed**. Codex + eng-review validated this comparison.

D-006 only pitted the LSTM-AE against IsolationForest. That panel was too narrow. Here we add a
**kNN-on-summary** and **GMM-on-summary** density detector (same 24-dim pooled stats IF uses) and
a **per-timestep position kNN** (the literal "how far from any known route?" test).

**Findings:** (1) kNN-on-summary **beats the AE** (0.707 vs 0.664) — the deep model doesn't earn
its complexity on synthetic; (2) the position-density "corridor" idea does **not** rescue
`zone_violation` (still ~0.57) — at 1–3 km the gaps between routes aren't empty. Both AE and the
frozen kNN are carried into the Phase-7 real-anomaly burn (07-eval-prep Layer 6); real anomalies
decide.

In [1]:
import sys, time
from pathlib import Path
import numpy as np, pandas as pd, torch
from sklearn.neighbors import NearestNeighbors
from sklearn.mixture import GaussianMixture
from sklearn.metrics import roc_auc_score

REPO = Path.cwd()
while not (REPO / "backend/core/preprocessing.py").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
from backend.core import split as sp
from backend.core.preprocessing import AE_FEATURES, SCALER_FEATURES, make_scaler, to_sequences, to_sequences_loss_mask
from backend.core.inject import make_eval_set
from backend.core.baseline import summary_features, IsolationForestBaseline, KNNSummaryBaseline
from backend.core import lstm_ae as ae

M=REPO/"backend/models/phase6"; SEED=42; rng=np.random.default_rng(SEED)
clean,meta=pd.read_parquet(M/"clean_df.parquet"),pd.read_parquet(M/"meta.parquet")
split=sp.split_by_monday(meta); train_df=sp.subset(clean,split.train_ids); val_df=sp.subset(clean,split.val_ids)
T=int(np.percentile(train_df.groupby("segment_id").size(),95))
scaler=make_scaler().fit(train_df[SCALER_FEATURES])
X_train,_,_=to_sequences(train_df,T,scaler); m_train=to_sequences_loss_mask(train_df,T)
vs=make_eval_set(clean,split.val_ids,scaler,T,seed=SEED,fold="val",inject_rate=0.5)
y,kind=vs.y,np.array(vs.kind)
OURTYPES=["zone_violation","altitude_high","sustained_loiter","final_approach_intercept","speed_spike"]
def per_type(score):
    norm=score[y==0]; out={}
    for k in OURTYPES:
        sel=kind==k
        if sel.sum(): out[k]=round(float(roc_auc_score(np.r_[np.zeros(len(norm)),np.ones(sel.sum())],np.r_[norm,score[sel]])),3)
    out["OVERALL"]=round(float(roc_auc_score(y,score)),3); return out
print(f"T={T}, val windows={len(y)}, anomalies={int(y.sum())}")

T=260, val windows=5942, anomalies=2971


In [2]:
results={}
agg=torch.load(str(M/"lstm_ae_best.pt"),map_location="cpu",weights_only=False)["extra"]["agg"]
model=ae.load_checkpoint(str(M/"lstm_ae_best.pt"))
results["LSTM-AE (small/mean)"]=per_type(ae.reconstruction_error(model,vs.X,vs.loss_mask,agg=agg))
results["IsolationForest"]=per_type(IsolationForestBaseline.fit(X_train,m_train,seed=SEED).anomaly_score(vs.X,vs.loss_mask))
results["kNN-summary (k=5)"]=per_type(KNNSummaryBaseline.fit(X_train,m_train,k=5).anomaly_score(vs.X,vs.loss_mask))
F_tr=summary_features(X_train,m_train); F_va=summary_features(vs.X,vs.loss_mask)
results["GMM-summary (8-diag)"]=per_type(-GaussianMixture(n_components=8,covariance_type="diag",random_state=SEED).fit(F_tr).score_samples(F_va))
# per-step POSITION kNN — the literal corridor test
li,lo=AE_FEATURES.index("lat"),AE_FEATURES.index("lon")
tr_pos=X_train[:,:,[li,lo]].reshape(-1,2)[(m_train.reshape(-1)>0)]
nn_pos=NearestNeighbors(n_neighbors=5).fit(tr_pos[rng.choice(len(tr_pos),size=min(80000,len(tr_pos)),replace=False)])
pos=np.zeros(len(vs.X))
for i in range(len(vs.X)):
    mv=vs.loss_mask[i]>0
    pos[i]=nn_pos.kneighbors(vs.X[i][mv][:,[li,lo]])[0][:,-1].mean() if mv.sum() else 0.0
results["position kNN (corridor)"]=per_type(pos)
pd.DataFrame(results).T[["zone_violation","altitude_high","sustained_loiter","final_approach_intercept","speed_spike","OVERALL"]]

,zone_violation,altitude_high,sustained_loiter,final_approach_intercept,speed_spike,OVERALL
LSTM-AE (small/mean),0.556,0.569,0.955,0.790,0.578,0.664
IsolationForest,0.500,0.545,0.937,0.739,0.543,0.625
kNN-summary (k=5),0.578,0.614,0.981,0.799,0.793,0.707
GMM-summary (8-diag),0.506,0.604,0.984,0.810,0.641,0.664
position kNN (corridor),0.570,0.484,0.472,0.650,0.464,0.532


## Conclusion

- **kNN-on-summary (0.707) > LSTM-AE (0.664) > IsolationForest (0.625)** on synthetic val — the
  same firewall. The deep model does **not** earn its complexity here (D-006's AE-vs-IF panel was
  too narrow; "DL track confirmed" is deferred to the Phase-7 real-anomaly test).
- The **position-density "corridor" test (0.53 overall, zone ~0.57)** confirms distance-to-known-
  routes does **not** separate a 1–3 km shift — the gaps aren't empty at that scale (only a hard
  APW polygon catches it, and that's APW's job under the D-010 reframe, not ours).
- **Caveat (codex):** kNN ignores temporal order/onset/semantics; its synthetic win may be partly
  bench-design fit and may not transfer to real go-arounds/emergencies (which can be summary-normal
  but order-abnormal). That is exactly why BOTH the AE and the **frozen** kNN
  (`backend/models/phase6/knn_train_summary.npy`) are carried, blind, into the Phase-7 burn —
  pre-registered in `07-eval-prep.md` Layer 6. Real anomalies decide.